In [4]:
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession

from utils.spark import create_spark, read_postgres
from utils.connector import postgres

def query(sql, params=None):
    with postgres() as pg:
        return pg.query(sql, params)


In [5]:
spark = SparkSession.builder \
    .master("spark://192.168.1.13:7077") \
    .appName("chembl_eda") \
    .config("spark.driver.host", "192.168.1.13") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()

jdbc_url = "jdbc:postgresql://192.168.1.13:5433/chembl_36"
properties = {
    "user": "chembl",
    "password": "chembl",
    "driver": "org.postgresql.Driver"
}


In [ ]:
sql_query = """
(
SELECT
    a.activity_id,
    a.assay_id,
    a.molregno,
    cs.canonical_smiles,
    a.standard_value,
    a.standard_units,
    a.standard_type,
    a.pchembl_value,
    a.action_type
FROM public.activities a
JOIN public.compound_structures cs
    ON a.molregno = cs.molregno
WHERE a.standard_value IS NOT NULL
) AS t
"""

df_activities = spark.read.jdbc(url=jdbc_url, table=sql_query, properties=properties)

df_activities.head()
df_activities.describe()

print("Liczba rekordów:", df_activities.count())

from pyspark.sql.functions import countDistinct
df_activities.select(countDistinct("canonical_smiles")).show()

output_path = "/home/patryk-kuszneruk/repos/wszi_2025_2026/parquets/activities.parquet"
df_activities.write.mode("overwrite").parquet(output_path)

26/01/15 19:21:43 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


# Liczba unikalnych molekuł

In [ ]:
n_rows = len(df_activities)
n_smiles = df_activities['canonical_smiles'].nunique()
print(f"Liczba rekordów: {n_rows}")
print(f"Liczba unikalnych SMILES: {n_smiles}")
print(f"Średnia liczba assayów na molekułę: {n_rows / n_smiles:.2f}")

# Duplikaty SMILES

In [ ]:
dups = df_activities.duplicated(subset=['canonical_smiles','standard_value'], keep=False)
print(f"Liczba duplikatów: {dups.sum()}")
df_activities[dups].head()

# Filtracja IC50 / nM

In [ ]:
df_filtered = df_activities[
    (df_activities['standard_type'] == 'IC50') &
    (df_activities['standard_units'] == 'nM')
].copy()

# Transformacja IC50 -> pIC50

In [ ]:
import numpy as np
df_filtered['pIC50'] = -np.log10(df_filtered['standard_value'] * 1e-9)
df_filtered[['standard_value','pIC50']].head()

In [ ]:
plt.hist(df_filtered['pIC50'], bins=50)
plt.xlabel("pIC50")
plt.ylabel("Liczba aktywności")
plt.show()

assays_per_smiles = df_filtered.groupby('canonical_smiles').size()
plt.boxplot(assays_per_smiles)
plt.ylabel("Liczba assayów / molekuła")
plt.show()